<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/03_window_5min_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB03 — Construção de Janelas Temporais de 5 Minutos

## 1. Contexto

Após a limpeza e normalização temporal realizada no NB02, este notebook transforma os eventos individuais em uma representação agregada no tempo.

O NB02 produz o arquivo `google_trace_clean.parquet`, contendo os registros temporalmente normalizados, ordenados e enriquecidos com `t_rel_us` e `hour`. A partir desse artefato, o NB03 discretiza a linha do tempo em janelas fixas de 5 minutos e constrói uma série temporal agregada, preservando continuidade, rastreabilidade e capacidade de interpretação.

Essa mudança de representação é essencial para o restante do pipeline, pois as etapas posteriores não trabalham diretamente com eventos individuais, mas com métricas consolidadas por janela temporal.

## 2. Objetivo

O objetivo principal deste notebook é construir a base temporal agregada em janelas fixas de 5 minutos.

Especificamente, este notebook busca:

1. Ler o arquivo `google_trace_clean.parquet`, produzido pelo NB02;
2. Confirmar a presença das colunas mínimas necessárias à agregação;
3. Validar a sanidade de `t_rel_us`;
4. Definir `bucket_id` a partir de janelas fixas de 5 minutos;
5. Calcular `bucket_start_us`;
6. Extrair informações de `resource_request`, quando disponíveis;
7. Agregar métricas operacionais por `bucket_id`;
8. Contabilizar eventos relevantes por tipo;
9. Construir uma base apenas com buckets observados;
10. Construir uma série contínua, incluindo janelas sem eventos;
11. Diagnosticar lacunas temporais;
12. Confirmar a presença do `bucket_id = 0`;
13. Persistir os artefatos canônicos e os artefatos de cenário;
14. Gerar o summary JSON da etapa.

## 3. Papel no Pipeline

O NB03 representa a passagem do nível de eventos individuais para o nível de série temporal agregada.

Ele não calcula criticidade, não detecta episódios críticos, não cria features temporais defasadas e não realiza modelagem. Sua função é estabelecer a unidade temporal regular que será usada nos notebooks seguintes.

| Notebook | Relação com o NB03 |
|---|---|
| NB02 | Fornece `google_trace_clean.parquet` |
| NB03 | Constrói buckets de 5 minutos e série contínua |
| NB04 | Usa a série temporal para cálculo de criticidade |
| NB05 | Usa a série com criticidade para engenharia de atributos |
| NB06 | Usa features e estados para formulação supervisionada |

## 4. Parâmetros da Execução

A execução utiliza janelas fixas de 5 minutos.

| Parâmetro | Valor | Interpretação |
|---|---:|---|
| `BUCKET_SEC` | 300 | Tamanho da janela em segundos |
| `BUCKET_US` | 300.000.000 | Tamanho da janela em microssegundos |
| `TOP_EVENTS` | `FAIL`, `SCHEDULE`, `FINISH`, `ENABLE`, `LOST`, `EVICT`, `KILL` | Eventos contabilizados individualmente |

O cenário executado é herdado do summary do NB02. Na execução de referência, o cenário é `keep_hour0`, com preservação do início da série temporal.

## 5. Métricas Agregadas

As métricas agregadas por janela incluem:

| Grupo | Métricas |
|---|---|
| Volume | `n_events`, `n_failed` |
| Diversidade operacional | `n_machines`, `n_collections` |
| Prioridade | `mean_priority` |
| Recursos solicitados | `mean_req_cpus`, `mean_req_mem` |
| Disponibilidade de requisições | `req_cpus_presence_rate`, `req_mem_presence_rate` |
| Eventos específicos | `event_FAIL_count`, `event_SCHEDULE_count`, `event_FINISH_count`, `event_ENABLE_count`, `event_LOST_count`, `event_EVICT_count`, `event_KILL_count` |

A base agregada contém apenas buckets observados. A série contínua preenche os buckets ausentes entre o menor e o maior `bucket_id`, preservando a regularidade temporal.

## 6. Artefatos Utilizados

Este notebook utiliza como entrada:

| Artefato | Origem | Finalidade |
|---|---|---|
| `google_trace_clean.parquet` | NB02 | Dataset limpo e temporalmente normalizado |
| `02_clean_normalize_summary.json` | NB02 | Metadados de cenário e decisão sobre `hour == 0` |

O notebook gera os seguintes artefatos:

| Artefato | Diretório | Finalidade |
|---|---|---|
| `window_5min_base.parquet` | `FEATURES_PATH` | Base agregada apenas com buckets observados |
| `window_5min_series.parquet` | `FEATURES_PATH` | Série contínua, incluindo janelas sem eventos |
| `window_5min_base_keep_hour0.parquet` | `FEATURES_PATH` | Base agregada específica do cenário |
| `window_5min_series_keep_hour0.parquet` | `FEATURES_PATH` | Série contínua específica do cenário |
| `03_window_5min_base_summary.json` | `REPORTS_PATH` | Summary canônico da agregação |
| `03_window_5min_base_summary_keep_hour0.json` | `REPORTS_PATH` | Summary específico do cenário |

## 7. Observações Metodológicas

A escolha de janelas de 5 minutos representa um compromisso entre granularidade temporal e estabilidade analítica.

A inclusão de janelas sem eventos é importante para preservar a continuidade da série e evitar que lacunas temporais sejam confundidas com ausência de observação.

Como o NB02 preservou `hour == 0`, espera-se que o NB03 preserve também o início da série, com presença de `bucket_id = 0`.

O arquivo `window_5min_series.parquet` deve ser considerado a saída canônica desta etapa para o NB04.

In [ ]:
# ============================================================
# NB03 — Construção de Janelas Temporais de 5 Minutos
# Pipeline PPCOMP_DM
#
# Escopo:
# - Ler o dataset limpo e normalizado produzido pelo NB02
# - Discretizar t_rel_us em janelas fixas de 5 minutos
# - Agregar métricas operacionais por bucket_id
# - Extrair métricas de resource_request quando disponível
# - Contabilizar eventos relevantes por tipo
# - Construir uma base agregada com buckets observados
# - Construir uma série contínua, incluindo janelas sem eventos
# - Persistir artefatos canônicos e de cenário
# - Persistir summaries JSON canônico e de cenário
#
# Importante:
# - Este notebook não calcula criticidade
# - Este notebook não detecta episódios críticos
# - Este notebook não realiza engenharia de atributos temporais
# - Este notebook não realiza modelagem
# ============================================================

# ─────────────────────────────────────────────────────────────
# BLOCO 0 — Bootstrap seguro
# ─────────────────────────────────────────────────────────────

from pathlib import Path
import os
import sys
import subprocess
import importlib
import json
import ast

# Monta o Google Drive apenas quando necessário.
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("[Bootstrap] Drive já montado.")

REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

# Garante o repositório local no Google Drive, clonando apenas se necessário.
if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)

# Garante prioridade do repositório no sys.path.
repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)

importlib.invalidate_caches()

from src.paths import PROCESSED_PATH, FEATURES_PATH, REPORTS_PATH, ensure_dirs

ensure_dirs()

def log(msg: str) -> None:
    """Imprime mensagens padronizadas desta etapa do pipeline."""
    print(f"[03_window_5min_base] {msg}")

# ─────────────────────────────────────────────────────────────
# BLOCO 1 — Parâmetros
# ─────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from IPython.display import display

# Janela temporal fixa de 5 minutos.
BUCKET_SEC = 5 * 60
BUCKET_US = BUCKET_SEC * 1_000_000

# Eventos contabilizados individualmente por bucket.
TOP_EVENTS = ["FAIL", "SCHEDULE", "FINISH", "ENABLE", "LOST", "EVICT", "KILL"]

# ─────────────────────────────────────────────────────────────
# BLOCO 2 — Metadados do NB02
# ─────────────────────────────────────────────────────────────

NB02_SUMMARY_FILE = REPORTS_PATH / "02_clean_normalize_summary.json"

nb02_summary = {}

if NB02_SUMMARY_FILE.exists():
    nb02_summary = json.loads(NB02_SUMMARY_FILE.read_text(encoding="utf-8"))
    log(f"Resumo do NB02 carregado: {NB02_SUMMARY_FILE}")
else:
    log("Resumo do NB02 não encontrado; seguindo sem metadados adicionais.")

SCENARIO_LABEL = nb02_summary.get("scenario_label", "unknown")
REMOVE_HOUR_ZERO = nb02_summary.get("remove_hour_zero", None)

# ─────────────────────────────────────────────────────────────
# BLOCO 3 — Leitura do dataset limpo
# ─────────────────────────────────────────────────────────────

CLEAN_PARQUET = PROCESSED_PATH / "google_trace_clean.parquet"

assert CLEAN_PARQUET.exists(), f"Arquivo não encontrado: {CLEAN_PARQUET}"

df = pd.read_parquet(CLEAN_PARQUET)

log(f"Shape entrada: {df.shape}")

required_cols = [
    "t_rel_us",
    "machine_id",
    "collection_id",
    "event",
    "failed",
    "priority",
]

missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Colunas ausentes: {missing}"

df["t_rel_us"] = pd.to_numeric(df["t_rel_us"], errors="coerce")
df = df.dropna(subset=["t_rel_us"]).copy()
df["t_rel_us"] = df["t_rel_us"].astype("int64")

assert (df["t_rel_us"] >= 0).all(), (
    "Foram encontrados valores negativos em t_rel_us."
)

df = df.sort_values(
    ["t_rel_us", "time"] if "time" in df.columns else ["t_rel_us"]
).reset_index(drop=True)

t_rel_min = int(df["t_rel_us"].min())
t_rel_max = int(df["t_rel_us"].max())

hour_min_input = int(df["hour"].min()) if "hour" in df.columns else None
hour_max_input = int(df["hour"].max()) if "hour" in df.columns else None

log(f"t_rel_us range: {t_rel_min}..{t_rel_max}")

if hour_min_input is not None and hour_max_input is not None:
    log(f"hour range entrada: {hour_min_input}..{hour_max_input}")

# ─────────────────────────────────────────────────────────────
# BLOCO 4 — Definição dos buckets
# ─────────────────────────────────────────────────────────────

# Cada evento é mapeado para uma janela de 5 minutos via divisão inteira.
df["bucket_id"] = (df["t_rel_us"] // BUCKET_US).astype("int64")
df["bucket_start_us"] = df["bucket_id"] * BUCKET_US

bucket_min_raw = int(df["bucket_id"].min())
bucket_max_raw = int(df["bucket_id"].max())
bucket_nunique_raw = int(df["bucket_id"].nunique())

log(f"bucket range bruto: {bucket_min_raw}..{bucket_max_raw}")
log(f"Buckets distintos brutos: {bucket_nunique_raw}")

if t_rel_min == 0:
    assert bucket_min_raw == 0, (
        "Com t_rel_us iniciando em 0, espera-se bucket_id inicial igual a 0."
    )
    log("Checkpoint OK: bucket inicial consistente com preservação do início da série.")

# ─────────────────────────────────────────────────────────────
# BLOCO 5 — Parse de resource_request
# ─────────────────────────────────────────────────────────────

def parse_dict(x):
    """
    Converte resource_request para dict quando possível.

    Suporta:
    - dict já pronto;
    - string JSON com aspas duplas;
    - string em formato de dict Python via ast.literal_eval.

    Retorna None quando não há conversão válida.
    """
    if isinstance(x, dict):
        return x

    if isinstance(x, str):
        s = x.strip()

        if not s:
            return None

        # Tentativa em JSON padrão.
        try:
            v = json.loads(s)
            return v if isinstance(v, dict) else None
        except Exception:
            pass

        # Fallback para string em formato de dict Python.
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, dict) else None
        except Exception:
            return None

    return None

if "resource_request" in df.columns:
    rr = df["resource_request"].map(parse_dict)

    df["req_cpus"] = rr.map(
        lambda d: d.get("cpus") if isinstance(d, dict) else np.nan
    )

    df["req_mem"] = rr.map(
        lambda d: d.get("memory") if isinstance(d, dict) else np.nan
    )
else:
    df["req_cpus"] = np.nan
    df["req_mem"] = np.nan

valid_cpu = int(df["req_cpus"].notna().sum())
valid_mem = int(df["req_mem"].notna().sum())

log(f"req_cpus válidos: {valid_cpu} ({valid_cpu / len(df):.4%})")
log(f"req_mem válidos: {valid_mem} ({valid_mem / len(df):.4%})")

df["has_req_cpus"] = df["req_cpus"].notna().astype("int64")
df["has_req_mem"] = df["req_mem"].notna().astype("int64")

# ─────────────────────────────────────────────────────────────
# BLOCO 6 — Agregações por bucket
# ─────────────────────────────────────────────────────────────

base = (
    df.groupby("bucket_id", as_index=False)
    .agg(
        bucket_start_us=("bucket_start_us", "min"),
        n_events=("event", "size"),
        n_failed=("failed", "sum"),
        n_machines=("machine_id", "nunique"),
        n_collections=("collection_id", "nunique"),
        mean_priority=("priority", "mean"),
        mean_req_cpus=("req_cpus", "mean"),
        mean_req_mem=("req_mem", "mean"),
        req_cpus_presence_rate=("has_req_cpus", "mean"),
        req_mem_presence_rate=("has_req_mem", "mean"),
    )
)

evt_counts = (
    df[df["event"].isin(TOP_EVENTS)]
    .groupby(["bucket_id", "event"], observed=False)
    .size()
    .unstack(fill_value=0)
)

for ev in TOP_EVENTS:
    if ev not in evt_counts.columns:
        evt_counts[ev] = 0

evt_counts = evt_counts[TOP_EVENTS].reset_index()
evt_counts = evt_counts.rename(
    columns={ev: f"event_{ev}_count" for ev in TOP_EVENTS}
)

base = base.merge(evt_counts, on="bucket_id", how="left")

count_cols = (
    ["n_events", "n_failed", "n_machines", "n_collections"]
    + [f"event_{ev}_count" for ev in TOP_EVENTS]
)

for c in count_cols:
    base[c] = base[c].fillna(0).astype("int64")

presence_rate_cols = ["req_cpus_presence_rate", "req_mem_presence_rate"]

for c in presence_rate_cols:
    base[c] = base[c].fillna(0.0).astype("float32")

base = base.sort_values("bucket_id").reset_index(drop=True)

base_rows = int(len(base))
base_bucket_min = int(base["bucket_id"].min())
base_bucket_max = int(base["bucket_id"].max())

log(f"Base agregada shape: {base.shape}")
log(f"Base bucket range: {base_bucket_min}..{base_bucket_max}")

# ─────────────────────────────────────────────────────────────
# BLOCO 7 — Série contínua
# ─────────────────────────────────────────────────────────────

# A série contínua preserva todos os buckets entre o menor e o maior bucket_id.
bmin = base_bucket_min
bmax = base_bucket_max

full = pd.DataFrame({"bucket_id": np.arange(bmin, bmax + 1, dtype=np.int64)})
full["bucket_start_us"] = full["bucket_id"] * BUCKET_US

series = full.merge(base, on=["bucket_id", "bucket_start_us"], how="left")

for c in count_cols:
    series[c] = series[c].fillna(0).astype("int64")

for c in presence_rate_cols:
    series[c] = series[c].fillna(0.0).astype("float32")

series = series.sort_values("bucket_id").reset_index(drop=True)

series_rows = int(len(series))
gap_buckets = int(series_rows - base_rows)
empty_window_ratio = float(gap_buckets / series_rows) if series_rows > 0 else 0.0

log(f"Série contínua shape: {series.shape}")
log(f"Gap buckets: {gap_buckets} ({empty_window_ratio:.4%})")

# ─────────────────────────────────────────────────────────────
# BLOCO 8 — Diagnósticos adicionais
# ─────────────────────────────────────────────────────────────

bucket0_present = bool((base["bucket_id"] == 0).any())

if bucket0_present:
    bucket0_row = base.loc[base["bucket_id"] == 0].iloc[0]

    bucket0_summary = {
        "bucket_id": int(bucket0_row["bucket_id"]),
        "bucket_start_us": int(bucket0_row["bucket_start_us"]),
        "n_events": int(bucket0_row["n_events"]),
        "n_failed": int(bucket0_row["n_failed"]),
        "n_machines": int(bucket0_row["n_machines"]),
        "n_collections": int(bucket0_row["n_collections"]),
    }
else:
    bucket0_summary = None

na_summary_series = {
    c: int(series[c].isna().sum())
    for c in ["mean_priority", "mean_req_cpus", "mean_req_mem"]
    if c in series.columns
}

log(f"Bucket 0 presente na base agregada: {bucket0_present}")

if bucket0_summary is not None:
    log(f"Resumo bucket 0: {bucket0_summary}")

# ─────────────────────────────────────────────────────────────
# BLOCO 9 — Persistência
# ─────────────────────────────────────────────────────────────

BASE_FILE = FEATURES_PATH / "window_5min_base.parquet"
SERIES_FILE = FEATURES_PATH / "window_5min_series.parquet"

BASE_SCENARIO_FILE = FEATURES_PATH / f"window_5min_base_{SCENARIO_LABEL}.parquet"
SERIES_SCENARIO_FILE = FEATURES_PATH / f"window_5min_series_{SCENARIO_LABEL}.parquet"

base.to_parquet(BASE_FILE, compression="snappy", index=False)
series.to_parquet(SERIES_FILE, compression="snappy", index=False)

base.to_parquet(BASE_SCENARIO_FILE, compression="snappy", index=False)
series.to_parquet(SERIES_SCENARIO_FILE, compression="snappy", index=False)

log(f"Base salva (canônica): {BASE_FILE}")
log(f"Série salva (canônica): {SERIES_FILE}")
log(f"Base salva (cenário): {BASE_SCENARIO_FILE}")
log(f"Série salva (cenário): {SERIES_SCENARIO_FILE}")

# ─────────────────────────────────────────────────────────────
# BLOCO 10 — Summary JSON
# ─────────────────────────────────────────────────────────────

summary = {
    "input_file": str(CLEAN_PARQUET),
    "scenario_label": SCENARIO_LABEL,
    "remove_hour_zero": REMOVE_HOUR_ZERO,
    "rows_in": int(len(df)),
    "t_rel_min": int(t_rel_min),
    "t_rel_max": int(t_rel_max),
    "hour_min_input": hour_min_input,
    "hour_max_input": hour_max_input,
    "bucket_id_min_raw": int(bucket_min_raw),
    "bucket_id_max_raw": int(bucket_max_raw),
    "bucket_id_nunique_raw": int(bucket_nunique_raw),
    "base_rows": int(base_rows),
    "series_rows": int(series_rows),
    "bucket_id_min": int(bmin),
    "bucket_id_max": int(bmax),
    "bucket_span": int(bmax - bmin + 1),
    "gap_buckets": int(gap_buckets),
    "empty_window_ratio": float(empty_window_ratio),
    "bucket0_present": bool(bucket0_present),
    "bucket0_summary": bucket0_summary,
    "avg_events_per_bucket": float(base["n_events"].mean()),
    "avg_failed_per_bucket": float(base["n_failed"].mean()),
    "req_cpu_valid_ratio": float(valid_cpu / len(df)),
    "req_mem_valid_ratio": float(valid_mem / len(df)),
    "top_events": TOP_EVENTS,
    "series_na_summary": na_summary_series,
    "output_base_file_canonical": str(BASE_FILE),
    "output_series_file_canonical": str(SERIES_FILE),
    "output_base_file_scenario": str(BASE_SCENARIO_FILE),
    "output_series_file_scenario": str(SERIES_SCENARIO_FILE),
}

SUMMARY_FILE = REPORTS_PATH / "03_window_5min_base_summary.json"
SUMMARY_SCENARIO_FILE = REPORTS_PATH / f"03_window_5min_base_summary_{SCENARIO_LABEL}.json"

SUMMARY_FILE.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

SUMMARY_SCENARIO_FILE.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

log(f"Resumo salvo (canônico): {SUMMARY_FILE}")
log(f"Resumo salvo (cenário): {SUMMARY_SCENARIO_FILE}")

# ─────────────────────────────────────────────────────────────
# BLOCO 11 — Visualização rápida
# ─────────────────────────────────────────────────────────────

print("\n=== RESUMO RÁPIDO ===")
print(f"Scenario label         : {SCENARIO_LABEL}")
print(f"Remove hour 0          : {REMOVE_HOUR_ZERO}")
print(f"Rows in                : {len(df)}")
print(f"Bucket range           : {bmin}..{bmax}")
print(f"Base rows              : {base_rows}")
print(f"Series rows            : {series_rows}")
print(f"Gap buckets            : {gap_buckets}")
print(f"Bucket 0 present       : {bucket0_present}")
print(f"Avg events per bucket  : {base['n_events'].mean():.4f}")
print(f"Avg failed per bucket  : {base['n_failed'].mean():.4f}")

print("\n=== HEAD BASE ===")
display(base.head())

print("\n=== HEAD SERIES ===")
display(series.head())

Mounted at /content/drive
[03_window_5min_base] Resumo do NB02 carregado: /content/drive/MyDrive/Mestrado/04-reports/02_clean_normalize_summary.json
[03_window_5min_base] Shape entrada: (405891, 23)
[03_window_5min_base] t_rel_us range: 0..2678923967375
[03_window_5min_base] hour range entrada: 0..744
[03_window_5min_base] bucket range bruto: 0..8929
[03_window_5min_base] Buckets distintos brutos: 8928
[03_window_5min_base] Checkpoint OK: bucket inicial consistente com preservação do início da série.
[03_window_5min_base] req_cpus válidos: 405117 (99.8093%)
[03_window_5min_base] req_mem válidos: 405117 (99.8093%)
[03_window_5min_base] Base agregada shape: (8928, 18)
[03_window_5min_base] Base bucket range: 0..8929
[03_window_5min_base] Série contínua shape: (8930, 18)
[03_window_5min_base] Gap buckets: 2 (0.0224%)
[03_window_5min_base] Bucket 0 presente na base agregada: True
[03_window_5min_base] Resumo bucket 0: {'bucket_id': 0, 'bucket_start_us': 0, 'n_events': 56452, 'n_failed': 37

,bucket_id,bucket_start_us,n_events,n_failed,n_machines,n_collections,mean_priority,mean_req_cpus,mean_req_mem,req_cpus_presence_rate,req_mem_presence_rate,event_FAIL_count,event_SCHEDULE_count,event_FINISH_count,event_ENABLE_count,event_LOST_count,event_EVICT_count,event_KILL_count
0,0,0,56452,37082,43394,1232,166.682739,0.018434,0.009607,0.986289,0.986289,37082,154,600,18616,0,0,0
1,2,600000000,28,9,28,19,208.464286,0.008100,0.016303,1.000000,1.000000,9,0,7,7,5,0,0
2,3,900000000,32,2,32,18,135.656250,0.009495,0.003928,1.000000,1.000000,2,0,19,4,6,0,1
3,4,1200000000,29,6,28,20,233.068966,0.012932,0.003717,1.000000,1.000000,6,0,6,6,11,0,0
4,5,1500000000,25,5,25,19,253.360000,0.011083,0.012531,1.000000,1.000000,5,0,4,8,8,0,0



=== HEAD SERIES ===


,bucket_id,bucket_start_us,n_events,n_failed,n_machines,n_collections,mean_priority,mean_req_cpus,mean_req_mem,req_cpus_presence_rate,req_mem_presence_rate,event_FAIL_count,event_SCHEDULE_count,event_FINISH_count,event_ENABLE_count,event_LOST_count,event_EVICT_count,event_KILL_count
0,0,0,56452,37082,43394,1232,166.682739,0.018434,0.009607,0.986289,0.986289,37082,154,600,18616,0,0,0
1,1,300000000,0,0,0,0,NaN,NaN,NaN,0.000000,0.000000,0,0,0,0,0,0,0
2,2,600000000,28,9,28,19,208.464286,0.008100,0.016303,1.000000,1.000000,9,0,7,7,5,0,0
3,3,900000000,32,2,32,18,135.656250,0.009495,0.003928,1.000000,1.000000,2,0,19,4,6,0,1
4,4,1200000000,29,6,28,20,233.068966,0.012932,0.003717,1.000000,1.000000,6,0,6,6,11,0,0


# Conclusão da Construção de Janelas Temporais

A etapa de discretização temporal e construção da série agregada foi concluída com sucesso.

O NB03 leu o arquivo `google_trace_clean.parquet`, produzido pelo NB02, e transformou os eventos individuais em uma série temporal regular com janelas fixas de 5 minutos.

Nesta etapa não foram calculados critérios de criticidade, episódios críticos, features temporais ou modelos preditivos. O foco permaneceu na construção da base agregada que servirá de entrada para o NB04.

## 8. Resultados da Execução

A execução atual apresentou os seguintes resultados:

| Item | Valor |
|---|---:|
| Registros de entrada | 405.891 |
| Colunas de entrada | 23 |
| Intervalo de `t_rel_us` | 0..2.678.923.967.375 |
| Faixa de `hour` na entrada | 0..744 |
| `bucket_id` mínimo bruto | 0 |
| `bucket_id` máximo bruto | 8.929 |
| Buckets distintos observados | 8.928 |
| Buckets na base agregada | 8.928 |
| Buckets na série contínua | 8.930 |
| Lacunas preenchidas | 2 |
| Proporção de janelas vazias | 0,0224% |
| Presença de `bucket_id = 0` | Sim |

Esses resultados indicam que a discretização em janelas de 5 minutos produziu uma série temporal quase totalmente ocupada, com apenas duas janelas sem eventos no intervalo analisado.

## 9. Coerência com o NB02

A execução refletiu corretamente a decisão metodológica adotada no NB02.

Como o NB02 preservou `hour == 0`, o NB03 manteve o início da série temporal e confirmou a presença de `bucket_id = 0`.

Esse comportamento foi validado pelo checkpoint interno do código, que exige `bucket_id = 0` quando `t_rel_us` inicia em zero.

A configuração herdada do NB02 foi:

| Parâmetro | Valor |
|---|---|
| `SCENARIO_LABEL` | `keep_hour0` |
| `REMOVE_HOUR_ZERO` | `False` |

## 10. Caracterização do Bucket Inicial

O bucket inicial apresentou volume extraordinariamente elevado em relação ao restante da série.

| Métrica | Valor |
|---|---:|
| `bucket_id` | 0 |
| `bucket_start_us` | 0 |
| `n_events` | 56.452 |
| `n_failed` | 37.082 |
| `n_machines` | 43.394 |
| `n_collections` | 1.232 |

Esse resultado confirma que a concentração inicial observada nos notebooks anteriores também se manifesta na agregação em janelas de 5 minutos.

O bucket 0 deve ser mantido em observação nas etapas seguintes, não como evidência automática de erro, mas como característica empírica relevante da base utilizada nesta execução.

## 11. Cobertura Temporal e Janelas Vazias

A comparação entre a base agregada e a série contínua indicou:

| Item | Valor |
|---|---:|
| Buckets observados na base | 8.928 |
| Buckets na série contínua | 8.930 |
| `gap_buckets` | 2 |
| `empty_window_ratio` | 0,0224% |

As duas lacunas foram preenchidas estruturalmente na série contínua, garantindo regularidade temporal para os notebooks posteriores.

Os valores ausentes nas métricas médias da série contínua estão associados a essas janelas sem eventos. Portanto, os `NaN` em `mean_priority`, `mean_req_cpus` e `mean_req_mem` não indicam falha da agregação, mas ausência de observações nas respectivas janelas.

## 12. Disponibilidade das Requisições de Recursos

A extração das variáveis derivadas de `resource_request` apresentou alta disponibilidade.

| Métrica | Valor |
|---|---:|
| `req_cpus` válidos | 405.117 |
| `req_mem` válidos | 405.117 |
| Razão válida de CPU solicitada | 99,8093% |
| Razão válida de memória solicitada | 99,8093% |

Esses resultados indicam que as variáveis `mean_req_cpus`, `mean_req_mem`, `req_cpus_presence_rate` e `req_mem_presence_rate` são adequadas para uso analítico nas etapas seguintes.

## 13. Artefatos Gerados

Nesta execução, foram gerados os seguintes artefatos:

| Artefato | Finalidade |
|---|---|
| `window_5min_base.parquet` | Base agregada apenas com buckets observados |
| `window_5min_series.parquet` | Série contínua, incluindo janelas sem eventos |
| `window_5min_base_keep_hour0.parquet` | Base agregada específica do cenário `keep_hour0` |
| `window_5min_series_keep_hour0.parquet` | Série contínua específica do cenário `keep_hour0` |
| `03_window_5min_base_summary.json` | Summary canônico da etapa |
| `03_window_5min_base_summary_keep_hour0.json` | Summary específico do cenário |

O arquivo `window_5min_series.parquet` deve ser considerado a saída canônica desta etapa para o NB04.

## 14. Relação com os Notebooks Posteriores

Com a série temporal contínua construída, o pipeline segue para o NB04, responsável pela detecção de janelas críticas e episódios críticos.

A partir da saída do NB03, o NB04 deverá:

1. consumir `window_5min_series.parquet`;
2. calcular a taxa de falha por janela;
3. definir o limiar exploratório de criticidade;
4. identificar janelas críticas;
5. agrupar janelas críticas contíguas em episódios;
6. persistir a série enriquecida com informações de criticidade.

## 15. Síntese Final

O NB03 cumpre adequadamente seu papel como etapa de construção da série temporal agregada.

A execução confirmou a consistência da discretização em janelas de 5 minutos, a preservação do início da série, a presença do `bucket_id = 0`, a alta cobertura temporal e a existência de apenas duas janelas vazias no intervalo analisado.

A base resultante é adequada para a detecção de criticidade no NB04 e mantém rastreabilidade com a decisão metodológica do NB02 de preservar `hour == 0`.

O NB03 deve, portanto, ser considerado fechado como etapa de agregação temporal da baseline.